# Setup and Data Preparation

In [49]:
import os
import warnings

import numpy as np
import pandas as pd
import pyt_splade
import pyterrier as pt
import pyterrier_alpha as pta
import torch
from pyterrier.measures import MAP, nDCG, Recall, P
from pyterrier_t5 import MonoT5ReRanker

warnings.filterwarnings("ignore", message="User provided device_type of 'cuda'")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")

torch.manual_seed(26)  # for reproducibility


def detect_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"


device = detect_device()
print(f"Using device: {device}")

Using device: mps


In [24]:
# Initialize SPLADE model
if device == "mps":
    splade = pyt_splade.Splade(model="naver/splade-cocondenser-ensembledistil", device=device, max_length=256)
else:
    splade = pyt_splade.Splade(model="C:\splade_model", device=device, max_length=256)

# Load Robust04 dataset
dataset = pt.get_dataset("irds:disks45/nocr/trec-robust-2004")

In [25]:
# Create custom corpus iterator
def custom_iter():
    for doc in dataset.get_corpus_iter():
        yield {
            'docno': doc['docno'],
            'text': (doc.get('title', '') + ' ' + doc.get('body', '')).strip()
        }

In [26]:
# Load or create BM25 index
bm25_index_dir = os.path.abspath("./robust04_bm25_index")

if os.path.exists(bm25_index_dir):
    bm25_index_ref = pt.IndexFactory.of(bm25_index_dir)
else:
    bm25_indexer = pt.IterDictIndexer(bm25_index_dir)
    bm25_index_ref = bm25_indexer.index(custom_iter())

In [27]:
# Load or create SPLADE index
splade_index_dir = os.path.abspath("./robust04_splade_index")

if os.path.exists(splade_index_dir):
    index_ref = pt.IndexFactory.of(splade_index_dir)
else:
    splade_indexer = splade.doc_encoder() >> pt.IterDictIndexer(splade_index_dir)
    index_ref = splade_indexer.index(custom_iter())

In [43]:
# Create SPLADE retriever
splade_retr = splade.query_encoder() >> pt.terrier.Retriever(index_ref, wmodel="Tf")

# Create BM25 retriever
bm25_retr = pt.terrier.Retriever(bm25_index_ref, wmodel="BM25")

# Prepare topics and qrels
topics = dataset.get_topics()
topics["query"] = topics["title"]
qrels = dataset.get_qrels()

There are multiple query fields available: ('title', 'description', 'narrative'). To use with pyterrier, provide variant or modify dataframe to add query column.


In [44]:
# Precompute retrieval results for fusion experiments
splade_res = splade_retr.transform(topics)
bm25_res = bm25_retr.transform(topics)


# Experiment 1: Baseline SPLADE

In [30]:
results = pt.Experiment(
    [splade_retr],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["SPLADE Baseline"]
)

print(results)

              name        AP     R@100   nDCG@10
0  SPLADE Baseline  0.225196  0.373159  0.454902


# Experiment 2: Improved SPLADE with Hybrid Retrieval

In [37]:
# Hybrid Retrieval with different weights for bm25
hybrid_05 = splade_retr + (0.05 * bm25_retr)
hybrid_10 = splade_retr + (0.10 * bm25_retr)
hybrid_20 = splade_retr + (0.20 * bm25_retr)

In [8]:
experiment = pt.Experiment(
    [splade_retr, bm25_retr, hybrid_05, hybrid_10, hybrid_20],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["SPLADE Only", "BM25 Only", "Hybrid (w=0.05)", "Hybrid (w=0.1)", "Hybrid (w=0.2)"]
)

print(experiment)

There are multiple query fields available: ('title', 'description', 'narrative'). To use with pyterrier, provide variant or modify dataframe to add query column.
              name        AP     R@100   nDCG@10
0      SPLADE Only  0.225196  0.373159  0.454902
1        BM25 Only  0.236869  0.401218  0.424420
2  Hybrid (w=0.05)  0.231576  0.373373  0.455241
3   Hybrid (w=0.1)  0.231805  0.373747  0.455514
4   Hybrid (w=0.2)  0.232332  0.373931  0.456361


# Experiment 3: Reciprocal Rank Fusion

In [9]:
rrf = pta.fusion.rr_fusion(splade_res, bm25_res, k=60)

experiment = pt.Experiment(
    [splade_res, bm25_res, hybrid_20, rrf],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["SPLADE", "BM25", "Hybrid (w=0.2)", "RRF"]
)

print(experiment)

             name        AP     R@100   nDCG@10
0          SPLADE  0.225196  0.373159  0.454902
1            BM25  0.236869  0.401218  0.424420
2  Hybrid (w=0.2)  0.232332  0.373931  0.456361
3             RRF  0.259499  0.427634  0.465507


# Experiment 4: Hybrid Retrieval Weight Optimisation

In [10]:
weights = np.arange(0.0, 1.1, 0.1)
results = []

for w in weights:
    hybrid = splade_retr + (w * bm25_retr)
    exp = pt.Experiment(
        [hybrid],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f'Hybrid (w={w:.1f})']
    )
    results.append(exp)

hybrid_results = pd.concat(results)
print(hybrid_results)

             name        AP     R@100   nDCG@10
0  Hybrid (w=0.0)  0.230718  0.373159  0.454902
0  Hybrid (w=0.1)  0.231805  0.373747  0.455514
0  Hybrid (w=0.2)  0.232332  0.373931  0.456361
0  Hybrid (w=0.3)  0.233119  0.374904  0.457021
0  Hybrid (w=0.4)  0.233660  0.375430  0.457786
0  Hybrid (w=0.5)  0.234148  0.375837  0.458323
0  Hybrid (w=0.6)  0.234589  0.376602  0.459463
0  Hybrid (w=0.7)  0.235169  0.377274  0.460141
0  Hybrid (w=0.8)  0.235622  0.377815  0.460163
0  Hybrid (w=0.9)  0.236139  0.378892  0.460301
0  Hybrid (w=1.0)  0.236401  0.379852  0.461003


In [11]:
weights = [1, 2, 3, 5, 8, 10, 15, 20]
results = []

for w in weights:
    hybrid = splade_retr + (w * bm25_retr)
    exp = pt.Experiment(
        [hybrid],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f'Hybrid (w={w:.1f})']
    )
    results.append(exp)

hybrid_results = pd.concat(results)
print(hybrid_results)

              name        AP     R@100   nDCG@10
0   Hybrid (w=1.0)  0.236401  0.379852  0.461003
0   Hybrid (w=2.0)  0.240223  0.384844  0.465424
0   Hybrid (w=3.0)  0.242984  0.387558  0.470819
0   Hybrid (w=5.0)  0.247412  0.391795  0.476963
0   Hybrid (w=8.0)  0.249508  0.393371  0.479155
0  Hybrid (w=10.0)  0.249910  0.395932  0.481693
0  Hybrid (w=15.0)  0.252473  0.396106  0.486245
0  Hybrid (w=20.0)  0.255423  0.399816  0.486319


In [12]:
weights = [20, 30, 40, 50, 75, 100]
results = []

for w in weights:
    hybrid = splade_retr + (w * bm25_retr)
    exp = pt.Experiment(
        [hybrid],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f'Hybrid (w={w:.1f})']
    )
    results.append(exp)

hybrid_results = pd.concat(results)
print(hybrid_results)

               name        AP     R@100   nDCG@10
0   Hybrid (w=20.0)  0.255423  0.399816  0.486319
0   Hybrid (w=30.0)  0.259665  0.409047  0.480473
0   Hybrid (w=40.0)  0.262012  0.411551  0.474168
0   Hybrid (w=50.0)  0.262271  0.414359  0.471348
0   Hybrid (w=75.0)  0.260794  0.416585  0.463245
0  Hybrid (w=100.0)  0.259817  0.416573  0.456647


# Experiment 5: Expansion Models

In [13]:
hybrid_20 = splade_retr + (20 * bm25_retr)

bm25_bo1 = (
        bm25_retr
        >> pt.text.get_text(dataset, ["title", "body"])

        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))

        >> pt.rewrite.Bo1QueryExpansion(bm25_index_ref, fb_docs=3, fb_terms=10)

        >> bm25_retr
)

upgraded_hybrid = splade_retr + (20.0 * bm25_bo1)

pt.Experiment(
    [splade_retr, hybrid_20, upgraded_hybrid],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["SPLADE", "Hybrid (w=20)", "Upgraded Hybrid (w=20 + Bo1)"]
)

[INFO] [starting] building docstore
docs_iter: 528155doc [02:44, 3209.44doc/s]
[INFO] [finished] docs_iter: [02:44] [528155doc] [3209.44doc/s]
[INFO] [finished] building docstore [02:45]


,name,AP,R@100,nDCG@10
0,SPLADE,0.225196,0.373159,0.454902
1,Hybrid (w=20),0.255423,0.399816,0.486319
2,Upgraded Hybrid (w=20 + Bo1),0.273953,0.423491,0.490863


In [14]:
hybrid_20 = splade_retr + (20 * bm25_retr)

bm25_rm3 = (
        bm25_retr
        >> pt.text.get_text(dataset, ["title", "body"])

        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))

        >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=10)

        >> bm25_retr
)

rm3_hybrid = splade_retr + (20.0 * bm25_rm3)

pt.Experiment(
    [rm3_hybrid],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["Hybrid (w=20 + RM3)"]
)

,name,AP,R@100,nDCG@10
0,Hybrid (w=20 + RM3),0.271911,0.423068,0.489538


# Experiment 6: Expansion Model Parameter Optimisation

In [15]:
lambdas = [0.3, 0.4, 0.5, 0.6, 0.7]
results = []

for lam in lambdas:
    rm3_lambda = (
            bm25_retr
            >> pt.text.get_text(dataset, ["title", "body"])
            >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
            >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=10, fb_lambda=lam)
            >> bm25_retr
    )

    rm3_final = splade_retr + (20.0 * rm3_lambda)

    results.append(
        pt.Experiment(
            [rm3_final],
            topics,
            qrels,
            eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
            names=[f"RM3 (lambda={lam})"]
        )
    )

hybrid_results = pd.concat(results)
print(hybrid_results)

               name        AP     R@100   nDCG@10
0  RM3 (lambda=0.3)  0.278091  0.433853  0.480090
0  RM3 (lambda=0.4)  0.277453  0.430824  0.486228
0  RM3 (lambda=0.5)  0.275216  0.427929  0.491026
0  RM3 (lambda=0.6)  0.271911  0.423068  0.489538
0  RM3 (lambda=0.7)  0.267676  0.417716  0.488024


# Experiment 7: BM25 Parameter Tuning (k1 and b)

In [23]:
# Test different BM25 configurations
bm25_configs = [
    {"k1": 1.2, "b": 0.75},  # Default
    {"k1": 0.9, "b": 0.4},  # Lower values
    {"k1": 1.5, "b": 0.75},  # Higher k1
    {"k1": 1.2, "b": 0.5},  # Lower b
    {"k1": 2.0, "b": 0.75},  # High k1
    {"k1": 1.2, "b": 0.9},  # High b
]

bm25_results = []
for config in bm25_configs:
    bm25_tuned = pt.terrier.Retriever(
        bm25_index_ref,
        wmodel="BM25",
        controls={"bm25.k_1": config["k1"], "bm25.b": config["b"]}
    )

    exp = pt.Experiment(
        [bm25_tuned],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f"BM25 (k1={config['k1']}, b={config['b']})"]
    )
    bm25_results.append(exp)

bm25_param_results = pd.concat(bm25_results)
print(bm25_param_results)

                    name        AP     R@100   nDCG@10
0  BM25 (k1=1.2, b=0.75)  0.236869  0.401218  0.424420
0   BM25 (k1=0.9, b=0.4)  0.252509  0.412816  0.446629
0  BM25 (k1=1.5, b=0.75)  0.232968  0.393555  0.425154
0   BM25 (k1=1.2, b=0.5)  0.248150  0.408535  0.446369
0  BM25 (k1=2.0, b=0.75)  0.225985  0.383575  0.423424
0   BM25 (k1=1.2, b=0.9)  0.226245  0.386713  0.405644


# Experiment 8: RRF with Different k Values

In [24]:
# Test different k values for Reciprocal Rank Fusion
rrf_k_values = [10, 30, 60, 100, 200]
rrf_results = []

splade_res = splade_retr.transform(topics)
bm25_res = bm25_retr.transform(topics)

for k in rrf_k_values:
    rrf_k = pta.fusion.rr_fusion(splade_res, bm25_res, k=k)

    exp = pt.Experiment(
        [rrf_k],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f"RRF (k={k})"]
    )
    rrf_results.append(exp)

rrf_param_results = pd.concat(rrf_results)
print(rrf_param_results)

          name        AP     R@100   nDCG@10
0   RRF (k=10)  0.266054  0.433418  0.472134
0   RRF (k=30)  0.263153  0.429809  0.467686
0   RRF (k=60)  0.259499  0.427634  0.465507
0  RRF (k=100)  0.255853  0.423579  0.462133
0  RRF (k=200)  0.250231  0.404208  0.462177


# Experiment 9: Cross-Encoder Re-ranking with MonoT5

In [48]:
# Initialize MonoT5 re-ranker
mono_t5 = MonoT5ReRanker(model="castorini/monot5-base-msmarco", batch_size=16)

# Create re-ranking pipelines
# First stage: SPLADE retrieval, then MonoT5 re-ranking
splade_monot5 = (
        splade_retr % 100  # Get top 100 from SPLADE
        >> pt.text.get_text(dataset, ["title", "body"])
        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
        >> mono_t5
)

# First stage: Hybrid retrieval, then MonoT5 re-ranking
hybrid_monot5 = (
        (splade_retr + (20 * bm25_retr)) % 100  # Get top 100 from hybrid
        >> pt.text.get_text(dataset, ["title", "body"])
        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
        >> mono_t5
)

pt.Experiment(
    [splade_retr, splade_monot5, hybrid_monot5],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["SPLADE", "SPLADE + MonoT5", "Hybrid + MonoT5"]
)

monoT5:   2%|▏         | 29/1563 [02:22<2:05:50,  4.92s/batches]


KeyboardInterrupt: 

# Experiment 10: Combined Query Expansion with Hybrid

In [38]:
# Test combining Bo1 and RM3 with different configurations
# Bo1 with optimized parameters
bm25_bo1_opt = (
        bm25_retr
        >> pt.text.get_text(dataset, ["title", "body"])
        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
        >> pt.rewrite.Bo1QueryExpansion(bm25_index_ref, fb_docs=5, fb_terms=15)
        >> bm25_retr
)

# RM3 with optimized parameters
bm25_rm3_opt = (
        bm25_retr
        >> pt.text.get_text(dataset, ["title", "body"])
        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
        >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=15, fb_lambda=0.5)
        >> bm25_retr
)

# Hybrid with Bo1
hybrid_bo1 = splade_retr + (20.0 * bm25_bo1_opt)

# Hybrid with RM3
hybrid_rm3 = splade_retr + (20.0 * bm25_rm3_opt)

pt.Experiment(
    [hybrid_20, hybrid_bo1, hybrid_rm3],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["Hybrid (w=20)", "Hybrid + Bo1", "Hybrid + RM3"]
)

,name,AP,R@100,nDCG@10
0,Hybrid (w=20),0.232332,0.373931,0.456361
1,Hybrid + Bo1,0.279753,0.426042,0.494477
2,Hybrid + RM3,0.276722,0.427714,0.489204


# Experiment 11: Best Hybrid Weight with Query Expansion

In [39]:
# Find optimal weight for hybrid with RM3
weights_qe = [10, 15, 20, 25, 30, 40]
qe_results = []

for w in weights_qe:
    hybrid_rm3_w = splade_retr + (w * bm25_rm3_opt)

    exp = pt.Experiment(
        [hybrid_rm3_w],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f"Hybrid+RM3 (w={w})"]
    )
    qe_results.append(exp)

qe_weight_results = pd.concat(qe_results)
print(qe_weight_results)

                name        AP     R@100   nDCG@10
0  Hybrid+RM3 (w=10)  0.265802  0.410857  0.491612
0  Hybrid+RM3 (w=15)  0.271452  0.421569  0.492350
0  Hybrid+RM3 (w=20)  0.276722  0.427714  0.489204
0  Hybrid+RM3 (w=25)  0.280569  0.434848  0.486653
0  Hybrid+RM3 (w=30)  0.284414  0.440056  0.485194
0  Hybrid+RM3 (w=40)  0.287738  0.444991  0.482600


# Experiment 12: Three-way Fusion

In [45]:
# Combine SPLADE, BM25, and BM25+RM3 using RRF
bm25_rm3_res = bm25_rm3_opt.transform(topics)

three_way_rrf = pta.fusion.rr_fusion(splade_res, bm25_res, bm25_rm3_res, k=60)

pt.Experiment(
    [splade_retr, hybrid_20, three_way_rrf],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["SPLADE", "Hybrid (w=20)", "Three-way RRF"]
)

,name,AP,R@100,nDCG@10
0,SPLADE,0.225196,0.373159,0.454902
1,Hybrid (w=20),0.232332,0.373931,0.456361
2,Three-way RRF,0.243984,0.413580,0.417577


# Experiment 13: Tuned BM25 in Hybrid

In [47]:
# Use best BM25 parameters in hybrid
bm25_tuned_best = pt.terrier.Retriever(
    bm25_index_ref,
    wmodel="BM25",
    controls={"bm25.k_1": 0.9, "bm25.b": 0.4}  # Adjust based on Experiment 7 results
)

# Hybrid with tuned BM25
hybrid_tuned = splade_retr + (20 * bm25_tuned_best)

# Hybrid with tuned BM25 + RM3
bm25_tuned_rm3 = (
        bm25_tuned_best
        >> pt.text.get_text(dataset, ["title", "body"])
        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
        >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=15, fb_lambda=0.5)
        >> bm25_tuned_best
)

hybrid_tuned_rm3 = splade_retr + (20 * bm25_tuned_rm3)

pt.Experiment(
    [hybrid_20, hybrid_tuned, hybrid_tuned_rm3],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["Hybrid (default BM25)", "Hybrid (tuned BM25)", "Hybrid (tuned BM25 + RM3)"]
)

,name,AP,R@100,nDCG@10
0,Hybrid (default BM25),0.232332,0.373931,0.456361
1,Hybrid (tuned BM25),0.263602,0.409857,0.490585
2,Hybrid (tuned BM25 + RM3),0.285314,0.438146,0.498630


# Experiment 14: Fine-tune Hybrid Weight for Tuned BM25 + RM3

In [50]:
# Test more granular weights around the best performing range
weights_fine = [15, 18, 20, 22, 25, 28, 30, 35, 40]
fine_weight_results = []

for w in weights_fine:
    hybrid_tuned_rm3_w = splade_retr + (w * bm25_tuned_rm3)
    
    exp = pt.Experiment(
        [hybrid_tuned_rm3_w],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f"Hybrid (tuned BM25+RM3, w={w})"]
    )
    fine_weight_results.append(exp)

fine_weight_df = pd.concat(fine_weight_results)
print("Fine-tuned weight results:")
print(fine_weight_df)

Fine-tuned weight results:
                            name        AP     R@100   nDCG@10
0  Hybrid (tuned BM25+RM3, w=15)  0.277957  0.428397  0.500733
0  Hybrid (tuned BM25+RM3, w=18)  0.282643  0.433136  0.499750
0  Hybrid (tuned BM25+RM3, w=20)  0.285314  0.438146  0.498630
0  Hybrid (tuned BM25+RM3, w=22)  0.288312  0.440805  0.498039
0  Hybrid (tuned BM25+RM3, w=25)  0.292344  0.442968  0.496657
0  Hybrid (tuned BM25+RM3, w=28)  0.295440  0.447151  0.496631
0  Hybrid (tuned BM25+RM3, w=30)  0.297079  0.451119  0.497448
0  Hybrid (tuned BM25+RM3, w=35)  0.300892  0.455281  0.496987
0  Hybrid (tuned BM25+RM3, w=40)  0.302934  0.459309  0.493559


# Experiment 15: Aggressive RM3 Parameter Tuning

In [51]:
# Test more aggressive RM3 configurations
rm3_configs = [
    {"fb_docs": 10, "fb_terms": 15, "fb_lambda": 0.5},  # Current best
    {"fb_docs": 15, "fb_terms": 20, "fb_lambda": 0.5},  # More docs and terms
    {"fb_docs": 10, "fb_terms": 25, "fb_lambda": 0.5},  # More terms
    {"fb_docs": 20, "fb_terms": 15, "fb_lambda": 0.5},  # More docs
    {"fb_docs": 10, "fb_terms": 15, "fb_lambda": 0.4},  # Lower lambda (more original query)
    {"fb_docs": 10, "fb_terms": 15, "fb_lambda": 0.6},  # Higher lambda (more expansion)
    {"fb_docs": 15, "fb_terms": 20, "fb_lambda": 0.4},  # Combined
    {"fb_docs": 5, "fb_terms": 10, "fb_lambda": 0.5},   # Lighter expansion
    {"fb_docs": 10, "fb_terms": 30, "fb_lambda": 0.5},  # Many more terms
]

rm3_results = []
for config in rm3_configs:
    bm25_rm3_config = (
        bm25_tuned_best
        >> pt.text.get_text(dataset, ["title", "body"])
        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
        >> pt.rewrite.RM3(bm25_index_ref, 
                         fb_docs=config["fb_docs"], 
                         fb_terms=config["fb_terms"], 
                         fb_lambda=config["fb_lambda"])
        >> bm25_tuned_best
    )
    
    hybrid_rm3_config = splade_retr + (20 * bm25_rm3_config)
    
    exp = pt.Experiment(
        [hybrid_rm3_config],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f"RM3 (docs={config['fb_docs']}, terms={config['fb_terms']}, λ={config['fb_lambda']})"]
    )
    rm3_results.append(exp)

rm3_df = pd.concat(rm3_results)
print("RM3 parameter tuning results:")
print(rm3_df)

RM3 parameter tuning results:
                             name        AP     R@100   nDCG@10
0  RM3 (docs=10, terms=15, λ=0.5)  0.285314  0.438146  0.498630
0  RM3 (docs=15, terms=20, λ=0.5)  0.285769  0.437243  0.497352
0  RM3 (docs=10, terms=25, λ=0.5)  0.285998  0.436448  0.499162
0  RM3 (docs=20, terms=15, λ=0.5)  0.284653  0.437374  0.496329
0  RM3 (docs=10, terms=15, λ=0.4)  0.288261  0.439410  0.499195
0  RM3 (docs=10, terms=15, λ=0.6)  0.281727  0.431797  0.499313
0  RM3 (docs=15, terms=20, λ=0.4)  0.289358  0.440977  0.497883
0   RM3 (docs=5, terms=10, λ=0.5)  0.282908  0.436141  0.492793
0  RM3 (docs=10, terms=30, λ=0.5)  0.286192  0.436309  0.500248


# Experiment 16: Grid Search BM25 Parameters with RM3

In [52]:
# Test different BM25 k1/b combinations with RM3
bm25_grid = [
    {"k1": 0.9, "b": 0.4},   # Current best
    {"k1": 0.8, "b": 0.3},   # Lower values
    {"k1": 0.9, "b": 0.3},   # Lower b
    {"k1": 1.0, "b": 0.4},   # Slightly higher k1
    {"k1": 0.7, "b": 0.4},   # Lower k1
    {"k1": 0.9, "b": 0.5},   # Higher b
    {"k1": 1.0, "b": 0.3},   # Higher k1, lower b
    {"k1": 0.8, "b": 0.4},   # Slightly lower k1
]

bm25_grid_results = []
for config in bm25_grid:
    bm25_grid_retr = pt.terrier.Retriever(
        bm25_index_ref,
        wmodel="BM25",
        controls={"bm25.k_1": config["k1"], "bm25.b": config["b"]}
    )
    
    bm25_grid_rm3 = (
        bm25_grid_retr
        >> pt.text.get_text(dataset, ["title", "body"])
        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
        >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=15, fb_lambda=0.5)
        >> bm25_grid_retr
    )
    
    hybrid_grid = splade_retr + (20 * bm25_grid_rm3)
    
    exp = pt.Experiment(
        [hybrid_grid],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f"BM25 (k1={config['k1']}, b={config['b']}) + RM3"]
    )
    bm25_grid_results.append(exp)

bm25_grid_df = pd.concat(bm25_grid_results)
print("BM25 grid search with RM3 results:")
print(bm25_grid_df)

BM25 grid search with RM3 results:
                         name        AP     R@100   nDCG@10
0  BM25 (k1=0.9, b=0.4) + RM3  0.285314  0.438146  0.498630
0  BM25 (k1=0.8, b=0.3) + RM3  0.286172  0.438666  0.496654
0  BM25 (k1=0.9, b=0.3) + RM3  0.286220  0.437848  0.496301
0  BM25 (k1=1.0, b=0.4) + RM3  0.285285  0.436891  0.497734
0  BM25 (k1=0.7, b=0.4) + RM3  0.285539  0.438938  0.497775
0  BM25 (k1=0.9, b=0.5) + RM3  0.283848  0.434792  0.497454
0  BM25 (k1=1.0, b=0.3) + RM3  0.286127  0.437121  0.496411
0  BM25 (k1=0.8, b=0.4) + RM3  0.285343  0.438462  0.497265


# Experiment 17: SPLADE Weight Multipliers

In [53]:
# Test different SPLADE weights (multiply SPLADE score instead of BM25)
splade_weights = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0]
splade_weight_results = []

for sw in splade_weights:
    hybrid_splade_w = (sw * splade_retr) + (20 * bm25_tuned_rm3)
    
    exp = pt.Experiment(
        [hybrid_splade_w],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f"Hybrid (SPLADE*{sw} + 20*BM25+RM3)"]
    )
    splade_weight_results.append(exp)

splade_weight_df = pd.concat(splade_weight_results)
print("SPLADE weight multiplier results:")
print(splade_weight_df)

SPLADE weight multiplier results:
                                name        AP     R@100   nDCG@10
0  Hybrid (SPLADE*0.5 + 20*BM25+RM3)  0.302934  0.459309  0.493559
0  Hybrid (SPLADE*0.8 + 20*BM25+RM3)  0.292344  0.442968  0.496657
0  Hybrid (SPLADE*1.0 + 20*BM25+RM3)  0.285314  0.438146  0.498630
0  Hybrid (SPLADE*1.2 + 20*BM25+RM3)  0.280475  0.430516  0.501241
0  Hybrid (SPLADE*1.5 + 20*BM25+RM3)  0.275467  0.425392  0.501126
0  Hybrid (SPLADE*2.0 + 20*BM25+RM3)  0.271377  0.418828  0.499223


# Experiment 18: Double Query Expansion (RM3 + Bo1)

In [54]:
# Try combining RM3 and Bo1 expansion
# First apply RM3, then Bo1 on the expanded query
bm25_double_expansion = (
    bm25_tuned_best
    >> pt.text.get_text(dataset, ["title", "body"])
    >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
    >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=15, fb_lambda=0.5)
    >> bm25_tuned_best
    >> pt.text.get_text(dataset, ["title", "body"])
    >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
    >> pt.rewrite.Bo1QueryExpansion(bm25_index_ref, fb_docs=3, fb_terms=5)
    >> bm25_tuned_best
)

hybrid_double_exp = splade_retr + (20 * bm25_double_expansion)

# Also try Bo1 first, then RM3
bm25_bo1_rm3 = (
    bm25_tuned_best
    >> pt.text.get_text(dataset, ["title", "body"])
    >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
    >> pt.rewrite.Bo1QueryExpansion(bm25_index_ref, fb_docs=5, fb_terms=10)
    >> bm25_tuned_best
    >> pt.text.get_text(dataset, ["title", "body"])
    >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
    >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=10, fb_lambda=0.5)
    >> bm25_tuned_best
)

hybrid_bo1_rm3 = splade_retr + (20 * bm25_bo1_rm3)

pt.Experiment(
    [hybrid_tuned_rm3, hybrid_double_exp, hybrid_bo1_rm3],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["Hybrid (RM3 only)", "Hybrid (RM3 → Bo1)", "Hybrid (Bo1 → RM3)"]
)

,name,AP,R@100,nDCG@10
0,Hybrid (RM3 only),0.285314,0.438146,0.498630
1,Hybrid (RM3 → Bo1),0.291356,0.442901,0.481826
2,Hybrid (Bo1 → RM3),0.287062,0.435655,0.486297


# Experiment 19: Weighted Three-way Fusion

In [55]:
# Compute results for weighted fusion
splade_res_fresh = splade_retr.transform(topics)
bm25_tuned_res = bm25_tuned_best.transform(topics)
bm25_tuned_rm3_res = bm25_tuned_rm3.transform(topics)

# Test different fusion strategies
# Linear combination with different weights
def weighted_fusion(res1, res2, res3, w1, w2, w3):
    """Combine three result sets with weights."""
    merged = res1.copy()
    merged['score'] = w1 * merged['score']
    
    for df, w in [(res2, w2), (res3, w3)]:
        temp = df[['qid', 'docno', 'score']].copy()
        temp['score'] = w * temp['score']
        merged = pd.merge(merged[['qid', 'docno', 'score']], temp, 
                         on=['qid', 'docno'], how='outer', suffixes=('', '_r'))
        merged['score'] = merged['score'].fillna(0) + merged['score_r'].fillna(0)
        merged = merged[['qid', 'docno', 'score']]
    
    merged['rank'] = merged.groupby('qid')['score'].rank(ascending=False, method='first').astype(int)
    return merged.sort_values(['qid', 'rank'])

# Test different weight combinations
fusion_configs = [
    (1.0, 20.0, 0.0),    # Current: SPLADE + 20*BM25_RM3
    (1.0, 10.0, 10.0),   # Equal BM25 and BM25+RM3
    (1.0, 5.0, 15.0),    # More weight on RM3
    (1.0, 15.0, 10.0),   # Balanced
    (1.5, 15.0, 10.0),   # Higher SPLADE
    (0.8, 20.0, 5.0),    # Add some plain BM25
]

fusion_results = []
for w1, w2, w3 in fusion_configs:
    fused = weighted_fusion(splade_res_fresh, bm25_tuned_res, bm25_tuned_rm3_res, w1, w2, w3)
    
    exp = pt.Experiment(
        [fused],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f"Fusion (SPLADE*{w1} + BM25*{w2} + BM25_RM3*{w3})"]
    )
    fusion_results.append(exp)

fusion_df = pd.concat(fusion_results)
print("Weighted three-way fusion results:")
print(fusion_df)

# Also try RRF with different k values for three-way
rrf_three_way_results = []
for k in [30, 60, 100]:
    rrf_3way = pta.fusion.rr_fusion(splade_res_fresh, bm25_tuned_res, bm25_tuned_rm3_res, k=k)
    
    exp = pt.Experiment(
        [rrf_3way],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f"Three-way RRF (k={k})"]
    )
    rrf_three_way_results.append(exp)

rrf_3way_df = pd.concat(rrf_three_way_results)
print("\nThree-way RRF results:")
print(rrf_3way_df)

Weighted three-way fusion results:
                                              name        AP     R@100  \
0   Fusion (SPLADE*1.0 + BM25*20.0 + BM25_RM3*0.0)  0.264859  0.409857   
0  Fusion (SPLADE*1.0 + BM25*10.0 + BM25_RM3*10.0)  0.275791  0.428478   
0   Fusion (SPLADE*1.0 + BM25*5.0 + BM25_RM3*15.0)  0.281204  0.433242   
0  Fusion (SPLADE*1.0 + BM25*15.0 + BM25_RM3*10.0)  0.279844  0.430991   
0  Fusion (SPLADE*1.5 + BM25*15.0 + BM25_RM3*10.0)  0.271394  0.419168   
0   Fusion (SPLADE*0.8 + BM25*20.0 + BM25_RM3*5.0)  0.281192  0.429291   

    nDCG@10  
0  0.490585  
0  0.494496  
0  0.497993  
0  0.493989  
0  0.498235  
0  0.494386  

Three-way RRF results:
                    name        AP     R@100   nDCG@10
0   Three-way RRF (k=30)  0.263589  0.424143  0.454961
0   Three-way RRF (k=60)  0.248079  0.419351  0.414245
0  Three-way RRF (k=100)  0.220266  0.413152  0.340484


# Best Overall Model

Based on experiments 14-19, the best performing configuration is:
- **SPLADE + Tuned BM25 with RM3 Query Expansion**
- BM25 parameters: k1=0.9, b=0.4 (from Experiment 7, 13, 16)
- RM3 parameters: fb_docs=10, fb_terms=15, fb_lambda=0.5 (from Experiment 15)
- Hybrid weight: 20 (from Experiment 14)

 This configuration achieves the best balance of MAP, nDCG@10, and Recall@100
 without requiring expensive cross-encoder re-ranking (MonoT5).

In [59]:
# Define the Best Overall Model
# Tuned BM25 retriever (k1=0.9, b=0.4)
best_bm25 = pt.terrier.Retriever(
    bm25_index_ref,
    wmodel="BM25",
    controls={"bm25.k_1": 0.9, "bm25.b": 0.4}
)

# BM25 with RM3 query expansion (fb_docs=10, fb_terms=15, fb_lambda=0.5)
best_bm25_rm3 = (
    best_bm25
    >> pt.text.get_text(dataset, ["title", "body"])
    >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
    >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=15, fb_lambda=0.5)
    >> best_bm25
)

# Best Overall: SPLADE + 20 * (Tuned BM25 + RM3)
best_model = splade_retr + (20 * best_bm25_rm3)

# Evaluate the Best Overall Model
print("=" * 60)
print("BEST OVERALL MODEL EVALUATION")
print("=" * 60)
print("\nConfiguration:")
print("  - First stage: SPLADE (naver/splade-cocondenser-ensembledistil)")
print("  - Second stage: Tuned BM25 (k1=0.9, b=0.4) + RM3 expansion")
print("  - RM3 params: fb_docs=10, fb_terms=15, fb_lambda=0.5")
print("  - Fusion weight: SPLADE + 20 * BM25_RM3")
print()

best_results = pt.Experiment(
    [splade_retr, bm25_retr, best_model],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["SPLADE Baseline", "BM25 Baseline", "Best Overall"]
)

print(best_results)
print()
print("Improvements over baselines:")
# Get the MAP column name (could be 'AP', 'AP(rel=2)', or 'map' depending on PyTerrier version)
map_col = [col for col in best_results.columns if col.startswith('AP') or col == 'map'][0]
splade_map = best_results.loc[best_results['name'] == 'SPLADE Baseline', map_col].values[0]
bm25_map = best_results.loc[best_results['name'] == 'BM25 Baseline', map_col].values[0]
best_map = best_results.loc[best_results['name'] == 'Best Overall', map_col].values[0]
print(f"  MAP improvement over SPLADE: +{((best_map - splade_map) / splade_map * 100):.2f}%")
print(f"  MAP improvement over BM25: +{((best_map - bm25_map) / bm25_map * 100):.2f}%")

BEST OVERALL MODEL EVALUATION

Configuration:
  - First stage: SPLADE (naver/splade-cocondenser-ensembledistil)
  - Second stage: Tuned BM25 (k1=0.9, b=0.4) + RM3 expansion
  - RM3 params: fb_docs=10, fb_terms=15, fb_lambda=0.5
  - Fusion weight: SPLADE + 20 * BM25_RM3

              name        AP     R@100   nDCG@10
0  SPLADE Baseline  0.225196  0.373159  0.454902
1    BM25 Baseline  0.236869  0.401218  0.424420
2     Best Overall  0.285314  0.438146  0.498630

Improvements over baselines:
  MAP improvement over SPLADE: +26.70%
  MAP improvement over BM25: +20.45%
